# 🔌 Utility Asset Detector

**One-click setup for detecting T&D utility infrastructure.**

Detects: Utility poles, transmission towers, insulators, transformers, cross-arms, damage (cracks, rust, woodpecker holes, lean, etc.)

**Instructions:** Click `Runtime → Run all` and wait ~5 minutes.

---

In [ ]:
#@title 1️⃣ Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f"\n✅ PyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 2️⃣ Install Dependencies (~2 min)
!pip install -q gradio pillow opencv-python pyyaml huggingface_hub

# Clone and install DART
!git clone -q https://github.com/mkturkcan/DART.git
%cd DART
!pip install -q -e .
%cd ..

# Clone utility-asset-detector
!git clone -q https://github.com/menonpg/utility-asset-detector.git
%cd utility-asset-detector

print("\n✅ Installation complete!")

In [ ]:
#@title 3️⃣ Download SAM3 Checkpoint (~1.7 GB)
from huggingface_hub import hf_hub_download
import os

if not os.path.exists("sam3.pt"):
    print("⏳ Downloading SAM3 checkpoint (2-3 min)...")
    hf_hub_download(
        repo_id="bodhicitta/sam3",
        filename="sam3.pt",
        local_dir="."
    )
    print("✅ Checkpoint downloaded!")
else:
    print("✅ Checkpoint exists")

print(f"   Size: {os.path.getsize('sam3.pt') / (1024**3):.2f} GB")

In [ ]:
#@title 4️⃣ Quick Test
import sys
sys.path.insert(0, ".")

from PIL import Image
import requests
from io import BytesIO

# Download test image (using raw GitHub which doesn't block)
print("Downloading test image...")
urls = [
    "https://images.unsplash.com/photo-1473341304170-971dccb5ac1e?w=400",
    "https://images.pexels.com/photos/2539462/pexels-photo-2539462.jpeg?w=400",
    "https://live.staticflickr.com/4066/4704340795_682a02d461_b.jpg",
]

test_image = None
for url in urls:
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200 and len(response.content) > 1000:
            test_image = Image.open(BytesIO(response.content))
            test_image.save("test_pole.jpg")
            print(f"✅ Test image downloaded")
            break
    except Exception as e:
        continue

if test_image is None:
    # Create a simple test image if all downloads fail
    print("Creating placeholder image...")
    test_image = Image.new('RGB', (400, 600), color='gray')
    test_image.save("test_pole.jpg")

# Initialize detector
print("Loading detector...")
from src.detector import UtilityAssetDetector

detector = UtilityAssetDetector(
    config="configs/assets.yaml",
    device="cuda"
)

result = detector.detect("test_pole.jpg")
print(f"\n✅ Detector ready!")
print(f"   Test: {result.total_structures} structures, {result.total_components} components")

In [ ]:
#@title 5️⃣ Launch Web UI 🚀
from app import create_ui

app = create_ui()
app.launch(share=True, debug=True)

---
## 📝 Direct Detection (Alternative to UI)

In [ ]:
#@title Upload Your Image
from google.colab import files
from src.detector import UtilityAssetDetector

print("Upload an image:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    detector = UtilityAssetDetector(config="configs/assets.yaml", device="cuda")
    result = detector.detect(filename)
    
    print(f"\n📊 Results:")
    print(f"Structures: {result.total_structures} | Components: {result.total_components} | Priority: {result.priority}")
    
    for struct in result.structures:
        print(f"\n📍 {struct.type} ({struct.confidence:.0%}) - {struct.condition.status}")
        if struct.condition.issues:
            print(f"   Issues: {', '.join(struct.condition.issues)}")
        for comp in struct.components:
            print(f"   • {comp.type}: {comp.condition.status}")

In [ ]:
#@title Download JSON Results
with open("result.json", "w") as f:
    f.write(result.to_json())
print(result.to_json())

from google.colab import files
files.download("result.json")